# Module 06 — RAG From First Principles

**Predict → Run → Observe → Explain → Break → Debug → Measure → Improve.** Build a framework-free retrieval pipeline and separate retrieval quality from generation quality.

## Objectives + architecture
- Document → chunk → embedding → index → retrieve → context → answer/abstain.
- Measure Recall@K, MRR, evidence coverage and latency.
- Prove tenant filtering is a security invariant.

Documents → chunks + metadata → deterministic embedding → exact vector search → evidence → grounded answer/abstention. Authorization happens before ranking/truncation.

In [ ]:
import sys
sys.path.insert(0, '..')
from app.models import Chunk
from app.index import VectorIndex


## BUILD — deterministic RAG baseline
Create a tiny multi-tenant corpus, index coherent chunks, and retrieve only authorized evidence.

In [ ]:
index=VectorIndex()
chunks=[Chunk('c1','d1','Vacation policy: employees receive 20 days of annual leave.',{'tenant_id':'a','section':'leave'}),Chunk('c2','d2','Security policy: production credentials require approval.',{'tenant_id':'a','section':'security'}),Chunk('c3','d3','Vacation policy: employees receive 25 days of annual leave.',{'tenant_id':'b','section':'leave'})]
for c in chunks:index.add(c)
results=index.search('How many annual leave days?',top_k=3,metadata_filter={'tenant_id':'a'})
print([(r.chunk.chunk_id,round(r.score,3)) for r in results])
assert results and all(r.chunk.metadata['tenant_id']=='a' for r in results)


## TRY — retrieval ablation
**TODO:** change `top_k` and query wording. Predict which evidence should rank first. Record Recall@K, context size and latency. Explain why filtering before ranking is part of correctness.

In [ ]:
for k in (1,2,3):
 print(k,[r.chunk.chunk_id for r in index.search('annual leave deadline',top_k=k,metadata_filter={'tenant_id':'a'})])


## BREAK — reproduce failures
1. Remove the tenant filter and show cross-tenant evidence becomes eligible.
2. Ask an unrelated question and define an abstention rule.
3. Add noisy/giant chunks and observe ranking degradation.
4. Pretend a generated answer is evidence; debug the missing citation boundary.

**Debug challenge:** classify the first failure as ingestion, retrieval, authorization, context assembly or generation.

In [ ]:
unsafe=index.search('annual leave days',top_k=3)
assert any(r.chunk.metadata['tenant_id']=='b' for r in unsafe)
print('BREAK reproduced: unfiltered retrieval admits another tenant.')


## MEASURE — Recall@K + MRR
Use labeled relevant chunk IDs. Extend the benchmark with evidence coverage, p50/p95 latency, token budget and cost/task.

In [ ]:
gold={'c1'}
ranked=[r.chunk.chunk_id for r in results]
recall_at_3=int(bool(set(ranked[:3])&gold))
rank=next((i+1 for i,c in enumerate(ranked) if c in gold),None)
mrr=1/rank if rank else 0.0
print({'Recall@3':recall_at_3,'MRR':mrr})


## SOLUTION + industry exercise
A correct baseline filters by authorization before ranking, keeps evidence IDs through context construction, and abstains when evidence coverage is insufficient.

**Exercise:** compare vector retrieval, deterministic lookup and long-context prompting. Then add a citation validator and abstention threshold.

**Mastery gate:** explain every RAG stage, reproduce a retrieval/security failure, calculate Recall@K/MRR and defend when RAG is the wrong architecture.